In [1]:
!pip install tensorflow opencv-python matplotlib

In [2]:
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [3]:
models = {
    'PET': load_model('PETmodelAlexNet-07-0.9985.hdf5'),
    'DAT': load_model('modeldatAlexnet-20-0.9958.hdf5'),
    'MRI': load_model('MRImodelAlexNet-94-0.9720.hdf5')
}

class_names = {0:'HC',1:'PD'}

C:\Users\JakirNiloy\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
def preprocess_image(img_path):
    
    img = cv2.imread(img_path)
    img = cv2.resize(img,(227,227))
    img = img.astype("float32") / 255.0
    
    img = np.expand_dims(img, axis=0)
    
    return img

In [7]:
for layer in models['PET'].layers:
    print(layer.name)

conv2d_25
max_pooling2d_15
batch_normalization_30
conv2d_26
max_pooling2d_16
batch_normalization_31
conv2d_27
batch_normalization_32
conv2d_28
batch_normalization_33
conv2d_29
max_pooling2d_17
batch_normalization_34
flatten_5
dense_10
dropout_5
batch_normalization_35
dense_11


In [9]:
import cv2
import numpy as np

img_path = "MRI_HC1.png"

img = cv2.imread(img_path)
img = cv2.resize(img,(227,227))

img_array = np.expand_dims(img, axis=0)
img_array = img_array.astype("float32")/255

AttributeError: 'str' object has no attribute 'show'

In [22]:
import numpy as np

# Assuming img_array is your input image with shape (227, 227, 3)
# If img_array already has a different shape, you may need to reshape it

# Create a dummy input with the correct shape
dummy_input = np.zeros((227, 227, 3))  # Create with expected dimensions

# Build the model by making a prediction
# Add only one batch dimension for the model
_ = models['PET'].predict(dummy_input[np.newaxis, ...])  # This creates shape (1, 227, 227, 3)

# For GradCAM, ensure img_array has the right dimensions
if len(img_array.shape) == 3:  # If img_array is (227, 227, 3)
    img_for_gradcam = img_array[np.newaxis, ...]  # Make it (1, 227, 227, 3)
else:
    # If img_array already has extra dimensions, reshape it correctly
    img_for_gradcam = img_array.reshape(-1, 227, 227, 3)

# Now you can use GradCAM
heatmap = gradcam(models['PET'], img_for_gradcam, "conv2d_29")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step


AttributeError: The layer sequential_5 has never been called and thus has no defined input.

In [17]:
def gradcam(model, img_array, last_conv_layer):

    # Run model once
    preds = model(img_array)

    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[model.get_layer(last_conv_layer).output, model.output]
    )

    with tf.GradientTape() as tape:

        conv_outputs, predictions = grad_model(img_array)

        class_idx = tf.argmax(predictions[0])
        loss = predictions[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap,0) / tf.reduce_max(heatmap)

    return heatmap.numpy()

In [15]:
models['PET'].get_layer("conv2d_29")

<Conv2D name=conv2d_29, built=True>

In [18]:
heatmap = gradcam(models['PET'], img_array, "conv2d_29")

AttributeError: The layer sequential_5 has never been called and thus has no defined input.

In [ ]:
def display_gradcam(img_path, heatmap, alpha=0.4):

    img = cv2.imread(img_path)
    img = cv2.resize(img,(227,227))

    heatmap = cv2.resize(heatmap,(227,227))
    heatmap = np.uint8(255 * heatmap)

    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed_img = heatmap * alpha + img

    plt.figure(figsize=(6,6))
    plt.imshow(cv2.cvtColor(superimposed_img.astype("uint8"), cv2.COLOR_BGR2RGB))
    plt.axis("off")

In [ ]:
img_path = "test_image.jpg"
dataset = "MRI"

img_array, pred_class = predict_image(img_path, dataset)

model = models[dataset]

heatmap = make_gradcam_heatmap(
    img_array,
    model,
    last_conv_layer_name="conv5"
)

display_gradcam(img_path, heatmap)